In [ ]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torchvision.models as models
import pandas as pd



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# Load labeled data with labels in the first column
def load_labeled_data(file_path, label_col=0):
    df = pd.read_excel(file_path, header=0)
    labels = df.iloc[:, label_col].values
    embeddings = df.iloc[:, 1:].values  # Embeddings start from the second column
    label_map = {'Racist': 0, 'Caste': 1, 'Offensive': 2, 'Sexist': 3}
    labels = torch.tensor([label_map[label] for label in labels], dtype=torch.long).to(device)
    embeddings = torch.tensor(embeddings, dtype=torch.float32).to(device)
    return embeddings, labels

# Load unlabeled data
def load_unlabeled_data(file_path):
    df = pd.read_excel(file_path, header=0)
    embeddings = torch.tensor(df.values, dtype=torch.float32).to(device)
    return embeddings

# Load the data files
train_embeddings, train_labels = load_labeled_data('Concatenated_Train_Embeddings.xlsx')
test_embeddings, test_labels = load_labeled_data('Concatenated_Test_Data.xlsx')
unlabeled_embeddings = load_unlabeled_data('Concatenated_Synthetic_Train_Data.xlsx')
synthetic_test_embeddings, synthetic_test_labels = load_labeled_data('Concatenated_Synthetic_Test_Data.xlsx')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torchvision.models as models

# Model Definition
class MultimodalResNet(nn.Module):
    def __init__(self, input_dim=1536, num_classes=4):
        super(MultimodalResNet, self).__init__()
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Modify the fully connected layers
        self.resnet.fc = nn.Sequential(
            nn.Linear(self.resnet.fc.in_features, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, 3, 32, 16)  # Example reshaping
        x = self.resnet(x)
        x = self.softmax(x)
        return x


In [ ]:
def train_model(model, criterion, optimizer, train_loader, num_epochs=50):
    model.train()
    train_loss_history = []
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        epoch_loss = running_loss / len(train_loader)
        train_loss_history.append(epoch_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    return train_loss_history

# Evaluation Function
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    report = classification_report(all_labels, all_preds, target_names=['Racist', 'Caste', 'Offensive', 'Sexist'])
    conf_matrix = confusion_matrix(all_labels, all_preds)
    return accuracy, f1, report, conf_matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score

# Plotting function for training loss curves
def plot_loss_curves(train_loss_teacher, train_loss_student, iteration):
    epochs = range(1, len(train_loss_teacher) + 1)
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, train_loss_teacher, label="Teacher Loss")
    plt.plot(epochs, train_loss_student, label="Student Loss")
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title(f'Teacher and Student Model Losses for Iteration {iteration}')
    plt.legend()
    plt.show()

# Training function without plotting confusion matrix inside
def train_teacher_student_model(train_loader, unlabeled_embeddings, train_embeddings, train_labels, num_epochs=30, num_iterations=2):
    # Initialize teacher and student models
    teacher_model = MultimodalResNet().to(device)
    student_model = MultimodalResNet().to(device)

    # Cross-entropy loss for labeled data
    criterion = nn.CrossEntropyLoss()

    # Lists to store losses for teacher and student models
    all_train_loss_teacher = []
    all_train_loss_student = []

    for iteration in range(num_iterations):
        print(f"Iteration {iteration + 1} of Teacher-Student training")

        # Train the teacher model on labeled data
        print("Training Teacher Model...")
        optimizer_teacher = optim.Adam(teacher_model.parameters(), lr=1e-4)
        train_loss_teacher = train_model(teacher_model, criterion, optimizer_teacher, train_loader, num_epochs)
        all_train_loss_teacher.append(train_loss_teacher)

        # Generate pseudo-labels for unlabeled data using the teacher model
        teacher_model.eval()
        with torch.no_grad():
            pseudo_labels = torch.argmax(teacher_model(unlabeled_embeddings), dim=1)

        # Create pseudo-labeled dataset for the student model
        pseudo_dataset = TensorDataset(unlabeled_embeddings, pseudo_labels)
        pseudo_loader = DataLoader(pseudo_dataset, batch_size=8, shuffle=True)

        # Combine labeled and pseudo-labeled data for the student model
        combined_train_loader = DataLoader(
            TensorDataset(
                torch.cat([train_embeddings, unlabeled_embeddings], dim=0),
                torch.cat([train_labels, pseudo_labels], dim=0)
            ),
            batch_size=8, shuffle=True
        )

        # Train the student model
        print("Training Student Model...")
        optimizer_student = optim.Adam(student_model.parameters(), lr=1e-4)
        student_model.train()

        train_loss_student = []
        for epoch in range(num_epochs):
            running_loss = 0.0
            for inputs, labels in combined_train_loader:
                optimizer_student.zero_grad()

                # Forward pass for labeled data
                outputs = student_model(inputs)
                loss_labeled = criterion(outputs, labels)

                # Compute KL divergence for pseudo-labeled data separately
                pseudo_inputs, _ = next(iter(pseudo_loader))
                student_outputs_unlabeled = student_model(pseudo_inputs)
                with torch.no_grad():
                    teacher_outputs_unlabeled = teacher_model(pseudo_inputs)

                kl_loss = F.kl_div(
                    F.log_softmax(student_outputs_unlabeled, dim=1),
                    F.softmax(teacher_outputs_unlabeled, dim=1),
                    reduction='batchmean'
                )

                # Total loss for student training
                total_loss = loss_labeled + kl_loss
                total_loss.backward()

                # Backward pass and optimization
                optimizer_student.step()
                running_loss += total_loss.item()

            # Calculate and store average student loss for the epoch
            epoch_loss = running_loss / len(combined_train_loader)
            train_loss_student.append(epoch_loss)
            print(f"Epoch {epoch + 1}/{num_epochs}, Student Loss: {epoch_loss:.4f}")

        # Store the student's loss per epoch for this iteration
        all_train_loss_student.append(train_loss_student)

        # Plot loss curves for the current iteration
        plot_loss_curves(train_loss_teacher, train_loss_student, iteration + 1)

        # Promote the student to be the new teacher for the next iteration
        teacher_model.load_state_dict(student_model.state_dict())
        print(f"End of iteration {iteration + 1}, student promoted to teacher.")

    return teacher_model, student_model, all_train_loss_teacher, all_train_loss_student


In [ ]:
# Create DataLoader objects
train_dataset = TensorDataset(train_embeddings, train_labels)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

test_dataset = TensorDataset(test_embeddings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Train the teacher and student models
teacher_model, student_model, all_train_loss_teacher, all_train_loss_student = train_teacher_student_model(train_loader, unlabeled_embeddings, train_embeddings, train_labels, num_epochs=100, num_iterations=3)


In [ ]:
import pandas as pd

# Function to evaluate model and display metrics and confusion matrix professionally
def evaluate_and_plot_confusion_matrix(model, data_loader, labels):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, true_labels in data_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(true_labels.cpu().numpy())

    # Compute metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    cm = confusion_matrix(all_labels, all_preds, labels=range(len(labels)))

    # Display metrics in a table format
    metrics_table = pd.DataFrame({
        "Metric": ["Accuracy", "Weighted F1 Score"],
        "Score": [accuracy, f1]
    })
    print("\nModel Evaluation Metrics:\n")
    print(metrics_table.to_string(index=False))

    # Plot confusion matrix
    print("\nConfusion Matrix:\n")
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap='Blues')
    plt.title("Confusion Matrix")
    plt.show()

    return accuracy, f1, cm

# Define label names
labels = ['Racist', 'Caste', 'Offensive', 'Sexist']

# Call evaluate_and_plot_confusion_matrix for the student model on the test set
accuracy, f1, cm = evaluate_and_plot_confusion_matrix(student_model, test_loader, labels)


In [ ]:
# Define the label names
labels = ['Racist', 'Caste', 'Offensive', 'Sexist']

# Call evaluate_and_plot_confusion_matrix for the student model on the test set
accuracy, f1, cm = evaluate_and_plot_confusion_matrix(student_model, test_loader, labels)

# Print the calculated metrics
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(cm)


In [ ]:
def train_teacher_student_model(train_loader, unlabeled_embeddings, train_embeddings, train_labels, num_epochs=30, num_iterations=2):
    # Initialize teacher and student models
    teacher_model = MultimodalResNet().to(device)
    student_model = MultimodalResNet().to(device)

    # Cross-entropy loss for labeled data
    criterion = nn.CrossEntropyLoss()

    # Lists to store losses for plotting
    train_loss_teacher = []
    train_loss_student = []  # New variable to store student labeled losses
    student_loss = []
    kl_loss = []

    for iteration in range(num_iterations):
        print(f"Iteration {iteration + 1} of Teacher-Student training")

        # Train the teacher model on labeled data
        print("Training Teacher Model...")
        optimizer_teacher = optim.Adam(teacher_model.parameters(), lr=1e-4)
        train_loss_teacher = train_model(teacher_model, criterion, optimizer_teacher, train_loader, num_epochs)

        # Generate pseudo-labels for unlabeled data using the teacher model
        teacher_model.eval()
        with torch.no_grad():
            pseudo_labels = torch.argmax(teacher_model(unlabeled_embeddings), dim=1)

        # Create pseudo-labeled dataset for the student model
        pseudo_dataset = TensorDataset(unlabeled_embeddings, pseudo_labels)
        pseudo_loader = DataLoader(pseudo_dataset, batch_size=8, shuffle=True)

        # Combine labeled and pseudo-labeled data for the student model
        combined_train_loader = DataLoader(
            TensorDataset(
                torch.cat([train_embeddings, unlabeled_embeddings], dim=0),
                torch.cat([train_labels, pseudo_labels], dim=0)
            ),
            batch_size=8, shuffle=True
        )

        # Train the student model
        print("Training Student Model...")
        optimizer_student = optim.Adam(student_model.parameters(), lr=1e-4)
        student_model.train()

        # Reset the losses for each iteration
        student_loss_epoch = []
        kl_loss_epoch = []
        train_loss_student_epoch = []  # To store student model labeled loss per epoch

        for epoch in range(num_epochs):
            running_loss = 0.0
            running_kl_loss = 0.0
            for inputs, labels in combined_train_loader:
                optimizer_student.zero_grad()

                # Forward pass for labeled data
                outputs = student_model(inputs)
                loss_labeled = criterion(outputs, labels)

                # Compute KL divergence for pseudo-labeled data
                pseudo_inputs, _ = next(iter(pseudo_loader))
                student_outputs_unlabeled = student_model(pseudo_inputs)
                with torch.no_grad():
                    teacher_outputs_unlabeled = teacher_model(pseudo_inputs)

                kl_loss_value = F.kl_div(
                    F.log_softmax(student_outputs_unlabeled, dim=1),
                    F.softmax(teacher_outputs_unlabeled, dim=1),
                    reduction='batchmean'
                )

                # Total loss for student training
                total_loss = loss_labeled + kl_loss_value
                total_loss.backward()

                # Backward pass and optimization
                optimizer_student.step()
                running_loss += loss_labeled.item()
                running_kl_loss += kl_loss_value.item()

            # Calculate average losses per epoch and append to lists
            epoch_loss = running_loss / len(combined_train_loader)
            epoch_kl_loss = running_kl_loss / len(combined_train_loader)
            student_loss_epoch.append(epoch_loss)
            kl_loss_epoch.append(epoch_kl_loss)
            train_loss_student_epoch.append(epoch_loss)  # Append student model's labeled loss per epoch

            print(f"Epoch {epoch + 1}/{num_epochs}, Student Model Loss: {epoch_loss:.4f}, KL Divergence Loss: {epoch_kl_loss:.4f}")

        # Extend main lists with epoch losses
        student_loss.extend(student_loss_epoch)
        kl_loss.extend(kl_loss_epoch)
        train_loss_student.extend(train_loss_student_epoch)  # Add student labeled losses

        # Promote the student to be the new teacher for the next iteration
        teacher_model.load_state_dict(student_model.state_dict())
        print(f"End of iteration {iteration + 1}, student promoted to teacher.")

    return teacher_model, student_model, train_loss_teacher, train_loss_student, kl_loss


In [ ]:
import matplotlib.pyplot as plt

def plot_training_curve(train_loss_history, title):
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss_history, label='Training Loss')
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(conf_matrix, title="Confusion Matrix"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Racist', 'Caste', 'Offensive', 'Sexist'],
                yticklabels=['Racist', 'Caste', 'Offensive', 'Sexist'])
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()


In [ ]:

# Plot training curves for both models
plot_training_curve(train_loss_teacher, title="Teacher Model Training Loss Curve")
plot_training_curve(train_loss_student, title="Student Model Training Loss Curve")

# Evaluate the student model on the original test data
accuracy, f1, report, conf = evaluate_model(student_model, test_loader)


In [ ]:
import pandas as pd

# Function to evaluate and print results
def evaluate_and_print_results(model, data_loader, data_description="Test Data"):
    # Evaluate the model
    accuracy, f1, report, conf_matrix = evaluate_model(model, data_loader)

    # Print evaluation scores
    print(f"Results for {data_description}:")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Classification Report:\n{report}")

    # Plot the confusion matrix
    plot_confusion_matrix(conf_matrix, title="Confusion Matrix")

    return accuracy, f1, report

# Evaluate the student model on the original test data (12 samples)
accuracy_12, f1_12, report_12 = evaluate_and_print_results(student_model, test_loader, "Test Set (12 samples)")

# Create DataLoader for synthetic test data (360 samples) if not already done
synthetic_test_dataset = TensorDataset(unlabeled_embeddings[:360])  # Adjust as needed
synthetic_test_loader = DataLoader(synthetic_test_dataset, batch_size=8, shuffle=False)

# Evaluate the student model on the synthetic test data (360 samples)
accuracy_360, f1_360, report_360 = evaluate_and_print_results(student_model, synthetic_test_loader, "Synthetic Test Set (360 samples)")

# Create a result comparison table
results_table = pd.DataFrame({
    "Metric": ["Accuracy", "F1 Score"],
    "Test Set (12 samples)": [f"{accuracy_12 * 100:.2f}%", f"{f1_12:.2f}"],
    "Synthetic Test Set (360 samples)": [f"{accuracy_360 * 100:.2f}%", f"{f1_360:.2f}"]
})

# Print the result table
print("\nComparison of Evaluation Results:")
print(results_table)


In [ ]:
''' def train_teacher_student_model(train_loader, unlabeled_embeddings, train_embeddings, train_labels, num_epochs=30, num_iterations=2):
    # Initialize teacher and student models
    teacher_model = MultimodalResNet().to(device)
    student_model = MultimodalResNet().to(device)

    # Cross-entropy loss for labeled data
    criterion = nn.CrossEntropyLoss()

    for iteration in range(num_iterations):
        print(f"Iteration {iteration + 1} of Teacher-Student training")

        # Train the teacher model on labeled data
        print("Training Teacher Model...")
        optimizer_teacher = optim.Adam(teacher_model.parameters(), lr=1e-4)
        train_loss_teacher = train_model(teacher_model, criterion, optimizer_teacher, train_loader, num_epochs)

        # Generate pseudo-labels for unlabeled data using the teacher model
        teacher_model.eval()
        with torch.no_grad():
            pseudo_labels = torch.argmax(teacher_model(unlabeled_embeddings), dim=1)

        # Create pseudo-labeled dataset for the student model
        pseudo_dataset = TensorDataset(unlabeled_embeddings, pseudo_labels)
        pseudo_loader = DataLoader(pseudo_dataset, batch_size=8, shuffle=True)

        # Combine labeled and pseudo-labeled data for the student model
        combined_train_loader = DataLoader(
            TensorDataset(
                torch.cat([train_embeddings, unlabeled_embeddings], dim=0),
                torch.cat([train_labels, pseudo_labels], dim=0)
            ),
            batch_size=8, shuffle=True
        )

        # Train the student model
        print("Training Student Model...")
        optimizer_student = optim.Adam(student_model.parameters(), lr=1e-4)
        student_model.train()

        for epoch in range(num_epochs):
            running_loss = 0.0
            for inputs, labels in combined_train_loader:
                optimizer_student.zero_grad()

                # Forward pass for labeled data
                outputs = student_model(inputs)
                loss_labeled = criterion(outputs, labels)

                # Compute KL divergence for pseudo-labeled data separately
                pseudo_inputs, _ = next(iter(pseudo_loader))
                student_outputs_unlabeled = student_model(pseudo_inputs)
                with torch.no_grad():
                    teacher_outputs_unlabeled = teacher_model(pseudo_inputs)

                kl_loss = F.kl_div(
                    F.log_softmax(student_outputs_unlabeled, dim=1),
                    F.softmax(teacher_outputs_unlabeled, dim=1),
                    reduction='batchmean'
                )

                # Total loss for student training
                total_loss = loss_labeled + kl_loss
                total_loss.backward()  # Removed retain_graph=True

                # Backward pass and optimization
                optimizer_student.step()
                running_loss += total_loss.item()

            epoch_loss = running_loss / len(combined_train_loader)
            print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")

        # Promote the student to be the new teacher for the next iteration
        teacher_model.load_state_dict(student_model.state_dict())
        print(f"End of iteration {iteration + 1}, student promoted to teacher.")

    return teacher_model, student_model, train_loss_teacher '''
